In [1]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import optuna
import joblib
import warnings
warnings.filterwarnings("ignore")
import xgboost as xgb
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score

c:\Projects\vlr-predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
splits = joblib.load("splits/6_windows_all_features.pkl")

In [3]:
def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi

In [4]:
def make_objective(X_train, y_train, X_val, y_val):
    def objective(trial):
        n_estimators     = trial.suggest_int('n_estimators', 100, 500)
        max_depth        = trial.suggest_int('max_depth', 3, 5)
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        subsample        = trial.suggest_float('subsample', 0.6, 1.0)
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)

        all_val_true, all_val_proba = [], []

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled   = scaler.transform(X_val)

        selector = VarianceThreshold(threshold=0.01)
        X_train_sel = selector.fit_transform(X_train_scaled)
        X_val_sel   = selector.transform(X_val_scaled)

        model = xgb.XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            eval_metric='logloss',
            random_state=42,
            n_jobs=1,  # n_jobs=-1 on model conflicts with n_jobs=-1 on study
        )
        model.fit(X_train_sel, y_train)

        val_proba = model.predict_proba(X_val_sel)[:, 1]
        all_val_true.append(y_val); all_val_proba.append(val_proba)
        logloss = log_loss(all_val_true, all_val_proba)
        return logloss

    return objective

In [ ]:
scores = []
parameters = []

for split in splits:
    X_train, y_train, X_val, y_val = split["X_train_raw"], split["y_train"], split["X_val_raw"], split["y_val"]

    objective = make_objective(X_train, y_train, X_val, y_val)
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=500, n_jobs=-1)
    params = study.best_params
    parameters.append(params)
    print("Best parameters:", params)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel   = selector.transform(X_val_scaled)

    model = xgb.XGBClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        eval_metric='logloss',
        random_state=42,
        n_jobs=1,  # n_jobs=-1 on model conflicts with n_jobs=-1 on study
    )
    model.fit(X_train_sel, y_train)

    val_pred = model.predict(X_val_sel)
    scores.append(roc_auc_score(y_val, val_pred))

print(f"Scores: {scores}")
print(f"Parameters: {parameters}")
print(f"Mean logloss: {np.mean(scores)}")

[I 2026-07-22 19:44:25,735] A new study created in memory with name: no-name-e1c962a9-ef65-4546-b152-94ea5eb19c4f
[I 2026-07-22 19:44:41,298] Trial 2 finished with value: 84.93212105986811 and parameters: {'n_estimators': 162, 'max_depth': 3, 'learning_rate': 0.10049327871644315, 'subsample': 0.7910871414510132, 'colsample_bytree': 0.8211846749444399, 'reg_alpha': 5.538372513548683, 'reg_lambda': 0.010453564920449939}. Best is trial 2 with value: 84.93212105986811.
[I 2026-07-22 19:44:41,781] Trial 13 finished with value: 84.90036751281828 and parameters: {'n_estimators': 189, 'max_depth': 3, 'learning_rate': 0.02601993029472487, 'subsample': 0.6085797819754525, 'colsample_bytree': 0.6324527636155142, 'reg_alpha': 3.31447354425371, 'reg_lambda': 0.946673748800973}. Best is trial 13 with value: 84.90036751281828.
[I 2026-07-22 19:44:42,637] Trial 14 finished with value: 116.67980659387358 and parameters: {'n_estimators': 130, 'max_depth': 5, 'learning_rate': 0.25230021839573924, 'subsam

In [ ]:
window_results = []
all_val_true, all_val_proba = [], []
all_test_true, all_test_proba = [], []

for i, (split, params) in enumerate(zip(splits, parameters)):
    X_train, y_train = split["X_train_raw"], split["y_train"]
    X_val, y_val = split["X_val_raw"], split["y_val"]
    X_test, y_test = split["X_test_raw"], split["y_test"]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    selector = VarianceThreshold(threshold=0.01)
    X_train_sel = selector.fit_transform(X_train_scaled)
    X_val_sel = selector.transform(X_val_scaled)
    X_test_sel = selector.transform(X_test_scaled)

    model = xgb.XGBClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        colsample_bytree=params['colsample_bytree'],
        reg_alpha=params['reg_alpha'],
        reg_lambda=params['reg_lambda'],
        eval_metric='logloss',
        random_state=42,
        n_jobs=1,  # n_jobs=-1 on model conflicts with n_jobs=-1 on study
    )
    model.fit(X_train_sel, y_train)

    val_proba = model.predict_proba(X_val_sel)[:, 1]
    test_proba = model.predict_proba(X_test_sel)[:, 1]

    window_results.append({
        "window": i,
        "n_val": len(y_val),
        "n_test": len(y_test),
        "val_logloss": log_loss(y_val, val_proba),
        "test_logloss": log_loss(y_test, test_proba),
        "test_acc": accuracy_score(y_test, (test_proba > 0.5).astype(int)),
    })

    all_val_true.append(y_val); all_val_proba.append(val_proba)
    all_test_true.append(y_test); all_test_proba.append(test_proba)

# --- Pool everything: concatenate, don't average ---
# Concatenating before scoring naturally weights each window by its
# sample count. Window 6 (252 matches) counts 252x, Window 2 (32) counts
# 32x — no manual weighting needed, unlike averaging six per-window scores.

pooled_val_true = np.concatenate(all_val_true)
pooled_val_proba = np.concatenate(all_val_proba)
pooled_test_true = np.concatenate(all_test_true)
pooled_test_proba = np.concatenate(all_test_proba)

pooled_test_logloss = log_loss(pooled_test_true, pooled_test_proba)
pooled_test_brier = brier_score_loss(pooled_test_true, pooled_test_proba)
pooled_test_acc = accuracy_score(pooled_test_true, (pooled_test_proba > 0.5).astype(int))

print(f"Pooled test log-loss: {pooled_test_logloss:.4f}")
print(f"Pooled test Brier:    {pooled_test_brier:.4f}")
print(f"Pooled test accuracy: {pooled_test_acc:.4f}  (n={len(pooled_test_true)})")

for r in window_results:
    print(r)


# --- Bootstrap CI, so you can tell if an iteration actually moved the needle ---

def bootstrap_logloss_ci(y_true, y_proba, n_boot=2000, ci=0.95, seed=0):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        scores[i] = log_loss(y_true[idx], y_proba[idx])
    lo, hi = np.percentile(scores, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return scores.mean(), lo, hi


mean_ll, lo, hi = bootstrap_logloss_ci(pooled_test_true, pooled_test_proba)
print(f"Pooled log-loss: {mean_ll:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

Pooled test log-loss: 0.6703
Pooled test Brier:    0.2365
Pooled test accuracy: 0.6240  (n=516)
{'window': 0, 'n_val': 198, 'n_test': 68, 'val_logloss': 0.6214685440063477, 'test_logloss': 0.6860423684120178, 'test_acc': 0.5882352941176471}
{'window': 1, 'n_val': 132, 'n_test': 32, 'val_logloss': 0.654501736164093, 'test_logloss': 0.7602555155754089, 'test_acc': 0.5625}
{'window': 2, 'n_val': 250, 'n_test': 48, 'val_logloss': 0.6395153403282166, 'test_logloss': 0.6524191498756409, 'test_acc': 0.625}
{'window': 3, 'n_val': 252, 'n_test': 68, 'val_logloss': 0.6295530796051025, 'test_logloss': 0.7140934467315674, 'test_acc': 0.6029411764705882}
{'window': 4, 'n_val': 320, 'n_test': 48, 'val_logloss': 0.643081545829773, 'test_logloss': 0.679000198841095, 'test_acc': 0.6666666666666666}
{'window': 5, 'n_val': 116, 'n_test': 252, 'val_logloss': 0.6272871494293213, 'test_logloss': 0.6445455551147461, 'test_acc': 0.6388888888888888}
Pooled log-loss: 0.6703  95% CI [0.6369, 0.7031]
